In [16]:
import kfp
import kfp.dsl as dsl
from kfp.dsl import component as component
from kfp import kubernetes

In [17]:
BASE_IMAGE = "traininghost/pipelinegpuimage:latest"

In [18]:
@dsl.component(base_image=BASE_IMAGE)
def train_export_model(trainingjobName: str, epochs: str, version: str):

    import subprocess
    import traceback
    import sys

    # gpu smoke check
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)

    def install(package):
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

    install('psutil')
    install('codecarbon')
    install('importlib_resources')

    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    import numpy as np
    import pandas as pd
    from sklearn.preprocessing import MinMaxScaler
    from featurestoresdk.feature_store_sdk import FeatureStoreSdk
    from modelmetricsdk.model_metrics_sdk import ModelMetricsSdk
    from codecarbon import OfflineEmissionsTracker
    import psutil
    import time
    import os

    blue = "\033[94m"
    reset = "\033[0m"

    fs_sdk = FeatureStoreSdk()
    mm_sdk = ModelMetricsSdk()

    SEQUENCE_LENGTH = 24
    FUTURE_STEPS = 1
    EPOCHS = epochs
    TRAIN_SPLIT_RATIO = 0.7
    NUM_FEATURES = 1
    LEARNING_RATE = 0.003
    BATCH_SIZE = 32

    def create_sequences(data, seq_length):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:(i + seq_length)])
            y.append(data[i + seq_length])
        return np.array(X), np.array(y)

    def rmse_loss(y_true, y_pred):
        err = y_true - y_pred
        mse = tf.reduce_mean(tf.square(err))
        rmse = tf.sqrt(mse)
        return rmse

    try:
        print("Job name is: ", trainingjobName)
        features = fs_sdk.get_features(trainingjobName, ['RRU.PrbUsedDl'])
        for index, row in features.iterrows():
            if (index + 1) % 30 == 0:
                continue
            features.drop(index, inplace=True)
        features.reset_index(drop=True, inplace=True)
        print("Dataframe shape:", features.shape)
        print("Dataframe info:")
        features.info()
        print("Dataframe head:")
        print(features.head())

        scaler = MinMaxScaler(feature_range=(0, 1))
        data_scaled = scaler.fit_transform(features.values.reshape(-1,1))

        X, y = create_sequences(data_scaled, SEQUENCE_LENGTH)

        train_size = int(len(X) * TRAIN_SPLIT_RATIO)
        X_train, X_test = X[:train_size], X[train_size:]
        y_train, y_test = y[:train_size], y[train_size:]

        model = Sequential([
            Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES)),
            LSTM(128, return_sequences=False),
            Dropout(0.2),
            Dense(50, activation='relu'),
            Dense(FUTURE_STEPS)
        ])

        optimizer = Adam(learning_rate=LEARNING_RATE)
        model.compile(optimizer=optimizer, loss=rmse_loss)
        model.summary()

        early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)

        try:
            EPOCHS = int(EPOCHS)
        except ValueError:
            print("Invalid epochs value. Using default of 50.")
            EPOCHS = 50

        tracker = OfflineEmissionsTracker(
            project_name="MCL Training",
            measure_power_secs=1,
            country_iso_code="KOR",
            save_to_file=False
        )
        tracker.start()
        try:
            start_cpu = psutil.cpu_percent(interval=1)

            history = model.fit(
                X_train, y_train,
                validation_data=(X_test, y_test),
                epochs=EPOCHS,
                shuffle=True,
                batch_size=BATCH_SIZE,
                verbose=1,
                callbacks=[early_stopping]
            )

            end_cpu = psutil.cpu_percent(interval=1)
        finally:
            tracker.stop()

        carbon_data = tracker.final_emissions_data
        training_time = carbon_data.duration
        ram_energy = carbon_data.ram_energy
        cpu_energy = carbon_data.cpu_energy
        gpu_energy = carbon_data.gpu_energy
        emissions = carbon_data.emissions

        print(f"{blue}Training Time:{reset} {training_time:.2f} seconds")
        print(f"{blue}RAM Energy:{reset} {ram_energy:.6f}kWh")
        print(f"{blue}CPU Energy:{reset} {cpu_energy:.6f}kWh")
        print(f"{blue}GPU Energy:{reset} {gpu_energy:.6f}kWh")
        print(f"{blue}CO2 Emissions:{reset} {emissions * 1000:.6f}gCO2")

        train_predict = model.predict(X_train)
        test_predict = model.predict(X_test)

        train_rmse = np.sqrt(np.mean((y_train - train_predict)**2, axis=0))
        test_rmse = np.sqrt(np.mean((y_test - test_predict)**2, axis=0))

        for i, feature in enumerate(features.columns):
            print(f'{blue}Train RMSE:{reset} {train_rmse[i]}, {blue}Test RMSE:{reset} {test_rmse[i]} ({feature})')

        model.export("./")
        model_size_kb = os.path.getsize('saved_model.pb') / 1024
        print(f"{blue}Model size:{reset} {model_size_kb:.2f} KB")

        data = {
            'metrics': [
                {f'{feature}_Train_RMSE': str(train_rmse[i])} for i, feature in enumerate(features.columns)
            ] + [
                {f'{feature}_Test_RMSE': str(test_rmse[i])} for i, feature in enumerate(features.columns)
            ] + [
                {'Training_Time': str(training_time)},
                {'RAM Energy': f'{ram_energy:.6f}kWh'},
                {'CPU Energy': f'{cpu_energy:.6f}kWh'},
                {'GPU Energy': f'{gpu_energy:.6f}kWh'},
                {'CO2_Emissions': f'{emissions:.6f}kgCO2'}
            ]
        }

        mm_sdk.upload_metrics(data, trainingjobName, version)
        mm_sdk.upload_model("./", trainingjobName, version)

    except Exception as e:
        print(f"Error in train_export_model: {str(e)}")
        print("Traceback:")
        print(traceback.format_exc())
        raise

@dsl.pipeline(
    name='Traffic Forecasting Pipeline',
    description='Deep Learning Based Traffic Forecasting Using LSTM.'
)
def time_series_pipeline(
    trainingjob_name: str, epochs: str, version: str):
    trainop = train_export_model(trainingjobName=trainingjob_name, epochs=epochs, version=version)
    trainop.set_gpu_limit(1)
    trainop.set_caching_options(False)
    kubernetes.set_image_pull_policy(trainop, "IfNotPresent")

In [19]:
pipeline_func = time_series_pipeline
file_name = "Traffic_Forecasting_pipeline"
kfp.compiler.Compiler().compile(pipeline_func, f'{file_name}.yaml')

In [20]:
import requests
pipeline_name = "Traffic_Forecasting_pipeline"
pipeline_file = file_name + '.yaml'
response = requests.post(
    "http://tm.traininghost:32002/pipelines/{}/upload".format(pipeline_name),
    files={'file': open(pipeline_file,'rb')}
)

if response.status_code == 200:
    print(f"Pipeline YAML file '{file_name}.yaml' has been generated and uploaded successfully.")
else:
    print(f"Failed to upload pipeline. Status code: {response.status_code}")
    print(f"Response content: {response.content}")

Pipeline YAML file 'Traffic_Forecasting_pipeline.yaml' has been generated and uploaded successfully.
